# 01 - Extração: SQL Server → MinIO (CSV)

Extrai **todas as tabelas** do database `LojaDB` do SQL Server e grava cada uma como CSV
no bucket **landing-zone** do MinIO (1 CSV por tabela).

**Pré-requisito:** notebook `00` executado.

## 1. Configuração

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if not (RAIZ / '.env').exists():
    raise FileNotFoundError('Arquivo .env não encontrado. Rode: cp .env.example .env')
load_dotenv(RAIZ / '.env', override=True)

import io
import pandas as pd
import pyodbc

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')

print(f'SQL Server: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')

SQL Server: localhost:1433/LojaDB
MinIO: http://localhost:9020 | Bucket: landing-zone


## 2. Conexão com o SQL Server

In [2]:
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;'
)
cursor = conn.cursor()
print(f'Conectado ao SQL Server [{DB_DATABASE}]')

Conectado ao SQL Server [LojaDB]


## 3. Listar as tabelas do banco

In [3]:
cursor.execute("""
    SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = 'dbo'
    ORDER BY TABLE_NAME
""")
tabelas = [row[0] for row in cursor.fetchall()]

print(f'{len(tabelas)} tabelas encontradas:')
for i, t in enumerate(tabelas, 1):
    cursor.execute(f'SELECT COUNT(*) FROM [{t}]')
    print(f'  {i:2d}. {t:<15} ({cursor.fetchone()[0]:>4} registros)')

2 tabelas encontradas:
   1. clientes        (   4 registros)
   2. pedidos         (   5 registros)


## 4. Criar o bucket landing-zone no MinIO

In [4]:
import boto3
from botocore.client import Config

s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] já existe')
except Exception:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [landing-zone] criado!
Buckets: ['landing-zone']


## 5. Extrair as tabelas → CSV no MinIO

In [5]:
print(f'Extraindo {len(tabelas)} tabelas...\n')
resultados = []

for tabela in tabelas:
    cursor.execute(f'SELECT * FROM [{tabela}]')
    colunas = [d[0] for d in cursor.description]
    df = pd.DataFrame.from_records(cursor.fetchall(), columns=colunas)

    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    csv_bytes = buffer.getvalue().encode('utf-8')
    chave = f'{tabela}.csv'

    s3_client.put_object(Bucket=LANDING_BUCKET, Key=chave,
                         Body=csv_bytes, ContentType='text/csv')

    resultados.append({'tabela': tabela, 'registros': len(df), 'colunas': len(df.columns)})
    print(f'  {tabela}.csv -> s3://{LANDING_BUCKET}/{chave} ({len(df)} registros)')

print(f'\nExtração concluída! {len(tabelas)} tabelas exportadas.')

Extraindo 2 tabelas...

  clientes.csv -> s3://landing-zone/clientes.csv (4 registros)
  pedidos.csv -> s3://landing-zone/pedidos.csv (5 registros)

Extração concluída! 2 tabelas exportadas.


## 6. Validação

In [6]:
resposta = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')
for obj in resposta.get('Contents', []):
    print(f'  {obj["Key"]:<20} {obj["Size"]:>6} bytes')

print()
print(pd.DataFrame(resultados).to_string(index=False))

Arquivos no bucket [landing-zone]:

  clientes.csv            136 bytes
  pedidos.csv             282 bytes

  tabela  registros  colunas
clientes          4        4
 pedidos          5        7


In [7]:
cursor.close()
conn.close()
print('Conexão SQL Server encerrada.')

Conexão SQL Server encerrada.
